# Data / Time Series & Recommendation Systems 실습 문제

각 문항의 `[문제]`와 `[요구사항]`을 읽고 코드의 `### 작성 필요 ###` 부분을 완성하세요.

## 문제 1. (a-1) MinMaxScaler 시계열 데이터 정규화 (fit_transform & transform)

[문제]
Q a-1) 시계열 학습 데이터(`train_data`)와 테스트 데이터(`test_data`)를 `MinMaxScaler`를 사용하여 [0, 1] 범위로 정규화합니다.

[요구사항]
- `MinMaxScaler(feature_range=(0, 1))`로 생성된 `scaler` 객체를 사용하시오.
- 학습 데이터(`train_data.values`)에는 최솟값·최댓값 통계량을 계산함과 동시에 변환을 수행하는 메서드를 호출하시오.
- 테스트 데이터(`test_data.values`)에는 미래 정보 누수(Data Leakage)를 방지하기 위해 학습 데이터에서 구한 통계량을 그대로 적용하여 변환만 수행하는 메서드를 호출하시오.
- ※ 사전 정의 컴포넌트: `MinMaxScaler`, DataFrame `train_data`, `test_data`

In [ ]:
# [코드]
# [0, 1] 범위로 정규화하는 MinMaxScaler 객체 생성
scaler = MinMaxScaler(feature_range=(0, 1))
# 학습 데이터의 최솟값/최댓값을 계산(fit)하고 [0, 1] 범위로 변환(transform)
train_scaled = ### 작성 필요 ###
# 데이터 누수 방지를 위해 테스트 데이터는 학습된 통계량으로 변환(transform)만 수행
test_scaled = ### 작성 필요 ###

## 문제 2. (a-2) Sliding Window 시계열 텐서 변환 (convert_data_into_tensors)

[문제]
Q a-2) 정규화된 1차원 시계열 배열 `data_seq`로부터 과거 `sequence_length` 시점의 입력 시퀀스(`features`)와 직후 시점의 정답값(`labels`) 쌍을 생성하여 PyTorch 텐서로 반환하는 `convert_data_into_tensors` 함수를 완성합니다.

[요구사항]
- 슬라이딩 윈도우 시작 인덱스 `i`에서 길이 `sequence_length`만큼의 연속 구간을 슬라이싱하여 `features` 리스트에 추가하시오.
- 입력 윈도우 바로 다음 시점(`i + sequence_length`)의 첫 번째 특성(`0`번 열) 값을 인덱싱하여 `labels` 리스트에 추가하시오.
- 변환된 NumPy 배열 `features`와 `labels`를 각각 `torch.float32` 데이터 타입의 텐서(`torch.tensor`)로 변환하시오.

In [ ]:
# [코드]
# 과거 50개 타임스텝을 참조하여 다음 시점을 예측하도록 윈도우 크기 설정
sequence_length = 50  # Number of time steps to look back

def convert_data_into_tensors(data_seq):
  features, labels = [], []
  # 전체 시계열 길이에서 윈도우 길이를 뺀 만큼 슬라이딩하며 샘플 생성
  for i in range(len(data_seq) - sequence_length):
      # i번째부터 sequence_length 길이만큼의 입력 구간 슬라이싱: (sequence_length, 1)
      features.append(### 작성 필요 ###)
      # 입력 구간 바로 다음 시점(i + sequence_length)의 0번 열 값을 타겟으로 추가
      labels.append(### 작성 필요 ###)
  # 리스트를 NumPy 배열로 일괄 변환
  features, labels = np.array(features), np.array(labels)

  # PyTorch 모델 학습을 위해 float32 텐서로 변환
  features = ### 작성 필요 ###
  labels = torch.tensor(labels, dtype=torch.float32)
  return features, labels

## 문제 3. (a-3) TensorDataset 및 DataLoader 미니배치 구성 (to_loader)

[문제]
Q a-3) 시계열 입력 텐서 `x`와 정답 텐서 `y`를 묶어 `TensorDataset`으로 만들고, 지정된 배치 크기와 셔플 옵션을 적용한 `DataLoader`를 반환하는 `to_loader` 함수를 완성합니다.

[요구사항]
- `torch.utils.data.TensorDataset`을 사용하여 입력 텐서 `x`와 타겟 텐서 `y`를 하나의 데이터셋 객체로 묶으시오.
- `torch.utils.data.DataLoader`에 `dataset`, `batch_size`, `shuffle`을 순서대로 전달하여 미니배치 로더를 반환하시오.

In [ ]:
# [코드]
batch_size = 32

def to_loader(x, y, batch_size, shuffle):
  # 입력 텐서 x와 라벨 텐서 y를 샘플 단위로 묶어 TensorDataset 생성
  dataset = ### 작성 필요 ###
  # 지정된 배치 크기와 셔플 여부에 따라 미니배치를 반환하는 DataLoader 생성
  return ### 작성 필요 ###

# 학습 데이터는 셔플을 활성화하고, 테스트 데이터는 순서를 유지하도록 로더 생성
train_loader = to_loader(X_train, y_train, batch_size, shuffle=True)
test_loader = to_loader(X_test, y_test, batch_size, shuffle=False)

## 문제 4. (a-4) 시계열 예측을 위한 LSTMModel 클래스 구현

[문제]
Q a-4) 다층 LSTM(`nn.LSTM`)과 선형 출력층(`nn.Linear`)으로 구성된 시계열 예측 모델 `LSTMModel` 클래스를 완성합니다.

[요구사항]
- `__init__`에서 `nn.LSTM`을 정의하되, 입력 텐서의 첫 번째 차원이 배치 크기(`(batch, seq, feature)`)가 되도록 `batch_first=True` 옵션을 반드시 지정하시오.
- LSTM의 은닉 차원(`hidden_size`)을 단일 예측값(`1`)으로 매핑하는 `nn.Linear` 레이어를 정의하시오.
- `forward`에서 `self.lstm(x)`를 호출하여 전체 시점의 출력 텐서 `out`을 얻고(은닉/셀 상태 튜플은 `_`로 무시), 선형 계층을 통과시킨 결과를 반환하시오.
- ※ 사전 정의 컴포넌트: `torch.nn as nn`

In [ ]:
# [코드]
input_size = X_train.shape[-1]
num_layers = 2
hidden_size = 64

class LSTMModel(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers):
        super(LSTMModel, self).__init__()
        # (B, L, input_size) 입력을 처리하는 다층 LSTM 레이어 정의 (batch_first=True 필수)
        self.lstm = ### 작성 필요 ###
        # 은닉 상태 벡터(hidden_size)를 1차원 예측값으로 변환하는 선형 레이어
        self.linear = ### 작성 필요 ###

    def forward(self, x):
        # LSTM 순전파 수행: out은 (B, L, hidden_size), 두 번째 반환값 (h_n, c_n)은 무시
        out, _ = ### 작성 필요 ###
        # 선형 계층을 통과시켜 (B, L, 1) 형상의 예측 텐서 반환
        return self.linear(out)

## 문제 5. (a-5) 시계열 모델 학습 및 평가 루프 (train)

[문제]
Q a-5) 시계열 예측 모델의 마지막 타임스텝 출력을 추출하여 MSE 손실을 계산하고 역전파로 가중치를 갱신하는 `train` 함수를 완성합니다.

[요구사항]
- 모델 출력 `model(batch_x)`(형상: `(B, seq_len, 1)`)에서 모든 배치의 **마지막 시점(`-1`)**, **0번 출력 채널(`0`)** 값만 슬라이싱하여 1차원 예측 텐서 `pred`를 구하시오.
- 손실 함수 `loss_fn`을 사용해 예측값 `pred`와 정답 `batch_y` 간의 오차를 계산하시오.
- 옵티마이저의 기울기 초기화, 역전파(`backward`), 파라미터 업데이트(`step`) 3단계를 순서대로 작성하시오.
- ※ 사전 정의 컴포넌트: `nn.MSELoss`, `optim.Adam`, `device`, `np`, `plt`

In [ ]:
# [코드]
def train(model, train_loader, test_loader):
  train_hist = []
  test_hist = []
  num_epochs = 10
  # 평균 제곱 오차(MSE) 손실 함수와 Adam 옵티마이저 설정
  loss_fn = nn.MSELoss(reduction='mean')
  optimizer = optim.Adam(model.parameters(), lr=1e-3)

  for epoch in range(num_epochs):
    total_train_loss = 0.0
    total_test_loss = 0.0

    # 학습 모드 활성화
    model.train()
    for batch_x, batch_y in train_loader:
      batch_x, batch_y = batch_x.to(device), batch_y.to(device)
      # (B, L, 1) 출력에서 마지막 타임스텝(-1)의 스칼라(0) 예측값만 추출 -> (B,)
      pred = ### 작성 필요 ###
      # 예측값과 정답 간의 MSE 손실 계산
      loss = ### 작성 필요 ###

      # 누적된 기울기 초기화
      ### 작성 필요 ###
      # 손실에 대한 역전파 수행
      ### 작성 필요 ###
      # 옵티마이저로 가중치 한 스텝 갱신
      ### 작성 필요 ###

      total_train_loss += loss.item()

    avg_loss = total_train_loss / len(train_loader)
    train_hist.append(avg_loss)

    # 평가 모드 전환 및 기울기 계산 비활성화
    model.eval()
    with torch.no_grad():
      for test_x, test_y in test_loader:
        test_x, test_y = test_x.to(device), test_y.to(device)
        test_pred = model(test_x)[:, -1, 0]
        test_loss = loss_fn(test_pred, test_y)
        total_test_loss += test_loss.item()

    avg_test_loss = total_test_loss / len(test_loader)
    test_hist.append(avg_test_loss)

## 문제 6. (a-6) 시계열 예측 성능 평가 지표 계산 (RMSE & MAPE)

[문제]
Q a-6) 학습된 시계열 모델로 테스트 데이터셋(`X_test`)의 마지막 시점 예측값을 추론하고, 실제값(`y_test`)과의 RMSE(평균 제곱근 오차) 및 MAPE(평균 절대 백분율 오차)를 계산하는 `test` 함수를 완성합니다.

[요구사항]
- 모델 출력 `y_hat`에서 마지막 타임스텝(`-1`)의 0번 특성 값을 슬라이싱하여 `test_predictions`에 저장하시오.
- `mean_squared_error(y_test, test_predictions)`의 결과에 `np.sqrt`를 취하여 `rmse`를 계산하시오.
- `mean_absolute_percentage_error(y_test, test_predictions)`를 호출하여 `mape`를 계산하시오.
- ※ 사전 정의 컴포넌트: `np`, `mean_squared_error`, `mean_absolute_percentage_error`, `device`

In [ ]:
# [코드]
def test(model, X_test, y_test):
  # 평가 모드 전환
  model.eval()
  with torch.no_grad():
    X_test = X_test.to(device)                     # (N, seq_len, 1)
    y_hat = model(X_test)                          # (N, seq_len, 1)
    # 마지막 시점(-1)의 스칼라 예측값(0) 추출
    test_predictions = ### 작성 필요 ###

  # scikit-learn 지표 계산을 위해 CPU NumPy 배열로 변환
  test_predictions = test_predictions.cpu().numpy()
  y_test = y_test.cpu().numpy()

  # MSE에 제곱근을 취하여 RMSE 계산
  rmse = ### 작성 필요 ###
  # 평균 절대 백분율 오차(MAPE) 계산
  mape = ### 작성 필요 ###

  return rmse, mape

## 문제 7. (a-7) 1D CNN 기반 시계열 모델 (Conv1DModel) 차원 변환 및 순전파

[문제]
Q a-7) 1차원 합성곱(`nn.Conv1d`)을 사용하여 시계열 특징을 추출하는 `Conv1DModel` 클래스를 완성합니다.

[요구사항]
- `__init__`에서 입력 채널 `input_size`, 출력 채널 `hidden_size`, `kernel_size=2`, `stride=1`을 갖는 `nn.Conv1d` 레이어를 정의하시오.
- `nn.Conv1d`는 `(Batch, Channels, Seq_len)` 형상의 입력을 요구하므로, `forward`에서 `(Batch, Seq_len, Channels)` 입력 `x`의 1번 차원과 2번 차원을 맞바꾸는(`transpose`) 연산을 합성곱 전후에 적용하시오.
- ※ 사전 정의 컴포넌트: `torch.nn as nn`

In [ ]:
# [코드]
class Conv1DModel(nn.Module):
  def __init__(self, input_size, hidden_size):
    super(Conv1DModel, self).__init__()
    # 입력 채널 input_size, 출력 채널 hidden_size, 커널 크기 2, 스트라이드 1인 1D 합성곱 정의
    self.conv1d = ### 작성 필요 ###
    # 은닉 채널(hidden_size)을 1차원 출력으로 매핑하는 선형 레이어
    self.fc = nn.Linear(hidden_size, 1)

  def forward(self, x):
    # (B, L, C) -> (B, C, L)로 차원 교환하여 Conv1d 입력 형식에 맞춤
    x = ### 작성 필요 ###
    x = self.conv1d(x)
    # (B, H, L') -> (B, L', H)로 다시 차원을 복원하여 Linear 계층 입력에 맞춤
    x = x.transpose(1, 2)
    return self.fc(x)

## 문제 8. (a-8) 다중 스텝 예측(Seq2Seq)을 위한 시퀀스 생성 (create_enc_dec_sequences)

[문제]
Q a-8) 과거 `sequence_length` 시점의 입력 시퀀스를 받아 미래 `target_len` 시점의 연속 구간을 예측하기 위한 Encoder-Decoder 학습용 시퀀스 생성 함수 `create_enc_dec_sequences`를 완성합니다.

[요구사항]
- 전체 데이터 길이 `len(data)`에서 입력 길이(`sequence_length`)와 타겟 길이(`target_len`)를 모두 확보할 수 있도록 `range`의 반복 상한을 작성하시오.
- `i`번째 윈도우의 입력 구간(`i`부터 `i + sequence_length`까지)과 다중 스텝 정답 구간(`i + sequence_length`부터 `i + sequence_length + target_len`까지)을 각각 슬라이싱하여 추가하시오.

In [ ]:
# [코드]
sequence_length = 50
target_len = 10

def create_enc_dec_sequences(data):
    features, labels = [], []
    # 입력 길이(sequence_length)와 출력 길이(target_len)를 모두 확보할 수 있는 범위까지 반복
    for i in range(### 작성 필요 ###):
        # 과거 sequence_length 구간을 인코더 입력으로 추가
        features.append(data[i:i + sequence_length])
        # 직후 target_len 구간을 디코더 타겟 시퀀스로 추가
        labels.append(### 작성 필요 ###)

    features = np.array(features, dtype=np.float32)
    labels = np.array(labels, dtype=np.float32)

    features = torch.tensor(features, dtype=torch.float32)
    labels = torch.tensor(labels, dtype=torch.float32)
    return features, labels

## 문제 9. (a-9) Encoder-Decoder RNN (RNNRNN) 자기회귀 디코딩 순전파

[문제]
Q a-9) 인코더(`EncoderRNN`)로 입력 시계열을 문맥 은닉 상태 `h`로 압축한 뒤, 디코더(`DecoderRNN`)를 `target_len`번 반복 호출하여 미래 시퀀스를 자기회귀(Autoregressive) 방식으로 생성하는 `RNNRNN` 모델을 완성합니다.

[요구사항]
- `self.encoder(source)`를 호출하여 인코더의 최종 은닉 상태 `h`를 구하시오.
- 디코더의 첫 타임스텝 입력(`input`)으로 `source`의 마지막 시점(`-1`) 벡터를 추출하고 시퀀스 차원(`dim=1`)을 추가(`unsqueeze(1)`)하시오.
- 루프 내부에서 `self.decoder(input, h)`를 호출하여 현재 스텝의 출력 `out`과 갱신된 은닉 상태 `h`를 얻고, 다음 스텝의 `input`에 현재 예측 출력 `out`을 대입하시오.
- `predictions` 리스트에 모인 텐서들을 타임스텝 차원(`dim=1`)으로 쌓아(`torch.stack`) 최종 텐서를 반환하시오.
- ※ 사전 정의 컴포넌트: `EncoderRNN`, `DecoderRNN`

In [ ]:
# [코드]
# [참고 컴포넌트 정의 - 세부 로직 비공개]
# ### 공개 제한 ###
class EncoderRNN(nn.Module):
  def __init__(self, input_size, hidden_size, num_layers):
    super(EncoderRNN, self).__init__()
    self.rnn = nn.RNN(input_size, hidden_size, num_layers, batch_first=True)

  def forward(self, x):
    _, h = self.rnn(x)
    return h

class DecoderRNN(nn.Module):
  def __init__(self, input_size, hidden_size, num_layers):
    super(DecoderRNN, self).__init__()
    self.rnn = nn.RNN(input_size, hidden_size, num_layers, batch_first=True)
    self.fc = nn.Linear(hidden_size, input_size)

  def forward(self, x, h):
    out, h = self.rnn(x, h)
    out = self.fc(out)
    return out, h

class RNNRNN(nn.Module):
  def __init__(self, input_size, hidden_size, num_layers):
    super(RNNRNN, self).__init__()
    self.encoder = EncoderRNN(input_size, hidden_size, num_layers)
    self.decoder = DecoderRNN(input_size, hidden_size, num_layers)

  def forward(self, source, target_len):
    # 인코더를 통해 입력 시계열로부터 문맥 은닉 상태 h 추출
    h = ### 작성 필요 ###
    predictions = []
    # 입력 시퀀스의 마지막 시점(-1) 값을 디코더의 첫 입력 (B, 1, C)로 설정
    input = ### 작성 필요 ###
    for t in range(target_len):
      # 디코더를 1스텝 실행하여 예측 출력 out과 갱신된 은닉 상태 h 획득
      out, h = ### 작성 필요 ###
      predictions.append(out.squeeze(1))
      # 현재 시점의 예측 출력 out을 다음 시점의 입력으로 사용 (자기회귀)
      input = out
    # 스텝별 예측 리스트를 시간 차원(dim=1)으로 쌓아 (B, target_len, C) 텐서 생성
    outputs = ### 작성 필요 ###

    return outputs

## 문제 10. (b-1) MovieLens 커스텀 Dataset 클래스 구현 (MovieLens)

[문제]
Q b-1) 사용자(`users`), 영화(`movies`), 평점(`ratings`) 배열을 받아 PyTorch `DataLoader`에 공급할 수 있는 커스텀 데이터셋 클래스 `MovieLens`를 완성합니다.

[요구사항]
- `__len__` 메서드에서 전체 샘플 수(`self.users`의 길이)를 반환하시오.
- `__getitem__` 메서드에서 인덱스 `item`에 해당하는 `users`, `movies`, `ratings` 값을 각각 `dtype=torch.long` 텐서로 변환하고 `.to(device)`로 디바이스에 올린 딕셔너리를 반환하시오.
- ※ 사전 정의 컴포넌트: `torch`, `device`

In [ ]:
# [코드]
class MovieLens:
  def __init__(self, users, movies, ratings):
    self.users = users
    self.movies = movies
    self.ratings = ratings

  def __len__(self):
    # 전체 데이터셋의 샘플 개수 반환
    return ### 작성 필요 ###

  def __getitem__(self,item):
    '''
    item = randomly selected indexes for (user,item) pairs
    '''
    users = self.users[item]
    movies = self.movies[item]
    ratings = self.ratings[item]
    # 임베딩 레이어 입력을 위해 torch.long 타입 텐서로 변환 후 디바이스로 이동
    return {'users': ### 작성 필요 ###,
            'movies': torch.tensor(movies, dtype = torch.long).to(device),
            'ratings': torch.tensor(ratings, dtype=torch.long).to(device)}

## 문제 11. (b-2) LabelEncoder 연속 인덱스 변환 및 층화 분할 (train_test_split)

[문제]
Q b-2) 비연속적인 원본 사용자 ID(`df.userId`)와 영화 ID(`df.movieId`)를 `LabelEncoder`로 `0`부터 시작하는 연속 정수로 변환하고, 평점(`df.rating.values`) 비율을 유지하며 학습/테스트 데이터프레임을 9:1로 분할합니다.

[요구사항]
- `lbl_user`와 `lbl_movie`의 `fit_transform` 메서드를 사용하여 `df.userId.values`와 `df.movieId.values`를 연속된 정수 인덱스로 변환하시오.
- `model_selection.train_test_split`을 호출하여 `df`를 `test_size=0.1`, `random_state=42`로 분할하되, 평점 분포가 동일하게 유지되도록 `stratify=df.rating.values`를 지정하시오.
- ※ 사전 정의 컴포넌트: `preprocessing`, `model_selection`, `MovieLens`, DataFrame `df`

In [ ]:
# [코드]
# 사용자 ID와 영화 ID를 0부터 시작하는 연속 정수로 매핑하기 위한 LabelEncoder 생성
lbl_user = preprocessing.LabelEncoder()
lbl_movie = preprocessing.LabelEncoder()

# 원본 userId와 movieId를 연속된 정수 인덱스로 변환
df.userId = ### 작성 필요 ###
df.movieId = lbl_movie.fit_transform(df.movieId.values)

# 평점(rating) 분포를 유지(stratify)하며 학습용과 테스트용 데이터프레임을 9:1 비율로 분할
df_train, df_test = ### 작성 필요 ###

train_dataset = MovieLens(users = df_train.userId.values, movies = df_train.movieId.values, ratings = df_train.rating.values)
test_dataset = MovieLens(users = df_test.userId.values, movies = df_test.movieId.values, ratings = df_test.rating.values)

## 문제 12. (b-3) Neural Collaborative Filtering (NCF) 모델 구현

[문제]
Q b-3) 사용자와 영화의 32차원 임베딩 벡터를 결합(Concatenate)한 뒤 2층 다층 퍼셉트론(MLP)에 통과시켜 평점을 예측하는 `Neural_Collaborative_Filtering` 클래스를 완성합니다.

[요구사항]
- `__init__`에서 사용자 수(`n_users`)와 영화 수(`n_movies`)를 각각 32차원 잠재 벡터로 매핑하는 `nn.Embedding` 레이어를 정의하시오.
- 두 32차원 임베딩을 결합한 64차원 벡터를 32차원으로 줄이는 `self.fc1`과, 이를 최종 1차원 평점으로 매핑하는 `self.fc2`를 `nn.Linear`로 정의하시오.
- `forward`에서 `user_embedding`과 `movie_embedding`을 특성 차원(`dim=1`)을 따라 `torch.cat`으로 결합하고, `fc1` → `relu` → `fc2`를 거쳐 반환하시오.

In [ ]:
# [코드]
class Neural_Collaborative_Filtering(nn.Module):
  def __init__(self, n_users, n_movies):
    super().__init__()
    # 사용자와 영화를 각각 32차원 잠재 벡터(latent vector)로 변환하는 임베딩 계층
    self.user_embedding = ### 작성 필요 ###
    self.movie_embedding = nn.Embedding(n_movies, 32)

    # 결합된 64차원(32 + 32) 벡터로부터 평점을 예측하는 2층 신경망(MLP)
    self.fc1 = ### 작성 필요 ###
    self.relu = nn.ReLU()
    self.fc2= nn.Linear(32,1)

  def forward(self, users, movies, ratings = None):
    user_embedding = self.user_embedding(users)
    movie_embedding = self.movie_embedding(movies)
    # 사용자 임베딩과 영화 임베딩을 특징 차원(dim=1)으로 연결 -> (B, 64)
    input_embedding = ### 작성 필요 ###

    hidden_feature = self.fc1(input_embedding)
    hidden_feature = self.relu(hidden_feature)

    # 최종 예측 평점 (B, 1) 산출
    output = self.fc2(hidden_feature)

    return output

## 문제 13. (b-4) NCF 평점 회귀 학습 및 RMSE 평가

[문제]
Q b-4) 테스트 데이터 로더(`test_loader`)를 순회하며 NCF 모델의 예측 평점과 실제 평점을 수집하고, `mean_squared_error`를 이용해 최종 RMSE(Root Mean Squared Error)를 계산합니다.

[요구사항]
- `with torch.no_grad():` 블록 내부에서 `model(batched_data['users'], batched_data['movies'])`를 호출하여 예측값을 얻으시오.
- 수집된 실제 평점 리스트 `target_rating_list`와 예측 평점 리스트 `model_output_list` 간의 평균 제곱 오차(`mean_squared_error`)를 계산하여 `mse`에 저장하시오.
- `mse`에 제곱근(`np.sqrt`)을 취하여 `rms`를 계산하시오.
- ※ 사전 정의 컴포넌트: `mean_squared_error`, `np`, `model`, `test_loader`

In [ ]:
# [코드]
from sklearn.metrics import mean_squared_error

model_output_list = []
target_rating_list = []

# 평가 모드 전환
model.eval()

with torch.no_grad():
  for i, batched_data in enumerate(test_loader):
    # 미니배치의 사용자 ID와 영화 ID를 입력하여 예측 평점 산출
    model_output = ### 작성 필요 ###
    # (B, 1) 텐서를 CPU NumPy로 옮긴 뒤 1번 축을 제거(squeeze)하여 리스트에 추가
    model_output_batch = model_output.cpu().numpy().squeeze(axis=1).tolist()
    model_output_list += (model_output_batch)

    target_rating = batched_data['ratings']
    target_rating_batch = target_rating.cpu().numpy().tolist()
    target_rating_list += target_rating_batch

# 전체 테스트 셋에 대한 평균 제곱 오차(MSE) 계산
mse = ### 작성 필요 ###
# MSE에 제곱근을 취하여 최종 RMSE 계산
rms = ### 작성 필요 ###
print(f"rms: {rms}")

## 문제 14. (b-5) 추천 시스템 Top-K 평가 지표 (Precision@K & Recall@K) 계산

[문제]
Q b-5) 사용자별 `(예측 평점, 실제 평점)` 리스트가 담긴 `user_est_true` 딕셔너리를 사용하여 상위 `k=10`개 추천에 대한 사용자별 `Precision@K`와 `Recall@K`를 계산합니다.

[요구사항]
- `user_ratings`를 예측 평점(`x[0]`) 기준 내림차순(`reverse=True`)으로 정렬하고, 상위 `k`개를 슬라이싱하여 `recommended_k`에 저장하시오.
- 상위 `k`개 추천 중 실제 관련 있는 아이템 개수(`n_real_relevant_in_top_k`)를 `10`으로 나누어 `precisions[user_id]`를 계산하시오.
- 실제 관련 아이템 총개수(`n_real_relevant`)가 존재할 때, `n_real_relevant_in_top_k`를 `n_real_relevant`로 나누어 `recalls[user_id]`를 계산하시오.
- ※ 사전 정의 컴포넌트: `user_est_true` (`dict[user_id, list[tuple[pred_rating, true_rating]]]`), `torch`

In [ ]:
# [코드]
with torch.no_grad():
  precisions = dict()
  recalls = dict()

  # 상위 추천 개수 k와 관련 아이템 판단 임계값(3.5점 이상) 설정
  k= 10
  threshold = 3.5 # relevant item criterion

  for user_id, user_ratings in user_est_true.items():

    # 사용자가 실제로 3.5점 이상 준 전체 관련 아이템 수 (Recall@k의 분모)
    n_real_relevant= sum((true_r >= threshold) for (_, true_r) in user_ratings)

    # 예측 평점(x[0])이 높은 순서대로 내림차순 정렬
    user_ratings.sort(key=lambda x: x[0], reverse =True)

    # 상위 k개 추천 결과 추출
    recommended_k = ### 작성 필요 ###

    # 상위 k개 추천 아이템 중 실제 평점이 3.5 이상인 적중(Hit) 개수 합산
    n_real_relevant_in_top_k = sum((true_r >= threshold) for (est, true_r) in recommended_k)

    # Precision@k 계산 (추천된 10개 중 실제 관련 아이템 비율)
    precisions[user_id] = ### 작성 필요 ###
    # Recall@k 계산 (전체 관련 아이템 중 추천된 상위 k개에 포함된 비율)
    if n_real_relevant:
      recalls[user_id] = ### 작성 필요 ###
    else:
      recalls[user_id] = 0

## 문제 15. (c-1) 사용자-아이템 이분 그래프 Edge Index 생성 (create_edge_index)

[문제]
Q c-1) 평점 데이터프레임(`df`)에서 평점이 `rating_threshold` 이상인 상호작용만 추출하여 PyTorch Geometric 형식의 `(2, E)` 텐서인 `edge_index`를 생성하는 `create_edge_index` 함수를 완성합니다.

[요구사항]
- 단일 그래프 내에서 사용자 노드(`0 ~ num_users-1`)와 영화 아이템 노드의 인덱스가 겹치지 않도록, 목적지(`dst`) 리스트에는 `row['movieId']`에 `num_users`를 더한 값을 추가하시오.
- 출발 노드 리스트 `src`와 도착 노드 리스트 `dst`를 묶어 `dtype=torch.long` 타입의 2차원 텐서로 반환하시오.
- ※ 사전 정의 컴포넌트: `torch`, 전역 변수 `num_users`, DataFrame `rating_df`

In [ ]:
# [코드]
def create_edge_index(df, rating_threshold=1.0):
    src, dst = [], []
    for _, row in df.iterrows():
        # 기준 평점 이상인 상호작용만 그래프의 엣지로 연결
        if row['rating'] >= rating_threshold:
            src.append(row['userId'])
            # 아이템 노드 인덱스가 사용자 노드 인덱스 뒤에 위치하도록 num_users 오프셋 합산
            dst.append(### 작성 필요 ###)
    # [출발 노드 리스트, 도착 노드 리스트]를 (2, E) 형상의 long 텐서로 반환
    return ### 작성 필요 ###

edge_index = create_edge_index(rating_df)

## 문제 16. (c-2) NGCF 그래프 컨볼루션 레이어 (NGCFLayer) 정규화 및 메시지 전달

[문제]
Q c-2) 사용자-아이템 이분 그래프에서 라플라시안 차수 정규화 계수(`norm`)를 계산하고, 이웃 노드와의 특징 친화도(Affinity)가 반영된 양방향 메시지를 집계하는 `NGCFLayer` 클래스를 완성합니다.

[요구사항]
- 각 엣지 `(src, dst)`에 대해 양 끝 노드의 차수(`deg[src]`, `deg[dst]`) 곱의 제곱근의 역수인 라플라시안 정규화 계수 `norm`을 계산하시오.
- 출발 노드(사용자)로 전달될 엣지 메시지 `edge_messages_for_src`를 `self.W1(dst_feat) + self.W2(dst_feat*src_feat)` 수식으로 계산하시오.
- `aggregated_messages.index_add_`를 사용하여 `src` 위치에 `edge_messages_for_src`를 누적 집계하시오.

In [ ]:
# [코드]
class NGCFLayer(nn.Module):
    def __init__(self, input_dim, output_dim, dropout=0.1):
        super().__init__()
        self.W1 = nn.Linear(input_dim, output_dim)
        self.W2 = nn.Linear(input_dim, output_dim)
        self.leaky_relu = nn.LeakyReLU(0.2)

    def forward(self, edge_index, node_features, user_num, item_num):
        src, dst = edge_index # src : user, dst : movie

        # 각 노드의 연결 차수(degree) 계산
        deg = torch.zeros(node_features.size(0), device=node_features.device)
        deg.index_add_(0, src, torch.ones_like(src, dtype=torch.float))
        deg.index_add_(0, dst, torch.ones_like(dst, dtype=torch.float))

        # 라플라시안 정규화 계수 1 / sqrt(deg(u) * deg(i)) 계산
        norm = ### 작성 필요 ###

        src_feat = node_features[src] # H_u
        dst_feat = node_features[dst] # H_i

        # 이웃 특징(W1)과 상호작용 요소별 곱 특징(W2)을 결합하여 메시지 생성
        edge_messages_for_src = ### 작성 필요 ###
        edge_messages_for_src *= norm.unsqueeze(1)

        edge_messages_for_dst = self.W1(src_feat) + self.W2(src_feat*dst_feat)
        edge_messages_for_dst *= norm.unsqueeze(1)

        # 노드별로 수신된 엣지 메시지 집계(index_add_) 및 자기 자신 변환(W1) 합산
        aggregated_messages = torch.zeros_like(node_features)
        ### 작성 필요 ###
        aggregated_messages[:user_num] += self.W1(node_features[:user_num])

        aggregated_messages.index_add_(0, dst, edge_messages_for_dst)
        aggregated_messages[user_num:] += self.W1(node_features[user_num:])

        aggregated_features = self.leaky_relu(aggregated_messages)
        return aggregated_features

## 문제 17. (c-3) NGCF 다층 표현 결합(Concatenation) 및 BPR Loss 구현

[문제]
Q c-3) 여러 `NGCFLayer`를 거치며 수집된 계층별 노드 표현(`layer_outputs`)을 결합하여 사용자/아이템 최종 임베딩을 추출하고, 선호 아이템 점수가 비선호 아이템 점수보다 높아지도록 학습하는 `bpr_loss` 메서드를 완성합니다.

[요구사항]
- `forward`에서 리스트 `layer_outputs`에 저장된 초기 임베딩과 각 레이어의 출력 텐서들을 마지막 특징 차원(`dim=-1`)으로 연결(`torch.concat`)하시오.
- 결합된 `final_features`에서 앞쪽 `self.num_users`개 행은 `user_features`로, 그 이후 행은 `item_features`로 슬라이싱하여 분리하시오.
- `bpr_loss`에서 긍정 쌍 점수와 부정 쌍 점수의 차이(`pos_scores - neg_scores`)에 `F.logsigmoid`를 취한 뒤 음의 평균(`-torch.mean(...)`)을 구하여 `loss`에 저장하시오.
- ※ 사전 정의 컴포넌트: `NGCFLayer`, `torch.nn.functional as F`

In [ ]:
# [코드]
class NGCF(nn.Module):
    def __init__(self, num_users, num_items, embedding_dim, layer_dims, dropout=0.1):
        super().__init__()
        self.num_users = num_users
        self.num_items = num_items
        self.embedding_dim = embedding_dim
        self.node_embeddings = nn.Embedding(self.num_users+self.num_items,self.embedding_dim)
        nn.init.xavier_uniform_(self.node_embeddings.weight)

        self.layers = nn.ModuleList([
            NGCFLayer(input_dim=(embedding_dim if i == 0 else layer_dims[i - 1]),
                      output_dim=layer_dims[i], dropout=dropout)
            for i in range(len(layer_dims))
        ])

    def forward(self, edge_index):
        node_features = self.node_embeddings.weight
        layer_outputs = [node_features]
        for layer in self.layers:
            node_features = layer(edge_index, node_features,self.num_users, self.num_items)
            layer_outputs.append(node_features)

        # 초기 임베딩과 각 레이어의 출력 임베딩을 마지막 차원(dim=-1)으로 결합
        final_features = ### 작성 필요 ###
        # 앞쪽 num_users개 행을 사용자 특징으로, 나머지를 아이템 특징으로 분리
        user_features = ### 작성 필요 ###
        item_features = final_features[self.num_users:]
        return user_features, item_features

    def bpr_loss(self, user_emb, pos_item_emb, neg_item_emb, reg_weight=1e-4):
        # 사용자-긍정 아이템 내적 점수 및 사용자-부정 아이템 내적 점수 계산
        pos_scores = torch.sum(user_emb * pos_item_emb, dim=1)
        neg_scores = torch.sum(user_emb * neg_item_emb, dim=1)
        # BPR 쌍별 순위 손실: -mean(log(sigmoid(pos_scores - neg_scores)))
        loss = ### 작성 필요 ###
        # L2 정규화 손실 추가
        reg_loss = reg_weight * (user_emb.norm(2).pow(2) + pos_item_emb.norm(2).pow(2)                                  + neg_item_emb.norm(2).pow(2)) / user_emb.size(0)
        return loss + reg_loss

## 문제 18. (c-4) NGCF 추천 랭킹 평가 함수 구현 (Top-K 추출 및 NDCG@K)

[문제]
Q c-4) 학습된 사용자/아이템 최종 특징 행렬을 내적하여 상위 `k`개 추천 아이템을 추출하고, 순위 가중치가 반영된 `DCG`, `IDCG`, `NDCG@K` 및 `Recall@K`, `Precision@K`를 계산하는 `evaluate` 함수를 완성합니다.

[요구사항]
- 전체 아이템 특징 행렬 `item_features`와 대상 사용자의 임베딩 `user_emb`를 행렬-벡터 곱(`torch.matmul`)하여 모든 아이템에 대한 선호도 점수 `scores`를 구하시오.
- `torch.topk(scores, k=k)`를 호출하여 상위 `k`개의 점수와 인덱스(`topk_indices`)를 추출하시오.
- 추천된 `rank`번째(0부터 시작) 아이템이 정답(`pos_items`)에 포함될 때, `1.0 / math.log2(rank + 2)`만큼 할인된 이득을 `dcg`에 누적하고, 최종적으로 `dcg / idcg`로 `ndcg_u`를 계산하시오.
- ※ 사전 정의 컴포넌트: `defaultdict`, `torch`, `math`, `np`, 전역 변수 `num_users`

In [ ]:
# [코드]
def evaluate(user_features, item_features, test_edge_index, k):
    user_pos_items = defaultdict(list)
    E_test = test_edge_index.size(1)
    for i in range(E_test):
        u = test_edge_index[0, i].item()
        it = test_edge_index[1, i].item() - num_users
        user_pos_items[u].append(it)

    recalls, precisions, ndcgs = [], [], []
    for user, pos_items in user_pos_items.items():
        user_emb = user_features[user]
        # 전체 아이템 특징과 사용자 임베딩의 내적으로 추천 점수 계산
        scores = torch.matmul(item_features, user_emb)
        # 점수가 가장 높은 상위 k개 아이템의 점수와 인덱스 추출
        topk_scores, topk_indices = ### 작성 필요 ###
        topk_indices = topk_indices.cpu().numpy().tolist()

        hits = 0
        dcg = 0.0
        idcg = 0.0
        n_pos = len(pos_items)

        for rank, item_idx in enumerate(topk_indices):
            if item_idx in pos_items:
                hits += 1
                # 적중한 아이템의 순위(rank)에 따른 로그 할인 이득(DCG) 누적
                dcg += ### 작성 필요 ###
        for rank in range(min(n_pos, k)):
            idcg += 1.0 / math.log2(rank + 2)

        recall_u = hits / n_pos
        precision_u = hits / k
        # 이상적인 최대 이득(IDCG) 대비 실제 DCG 비율로 NDCG 계산
        ndcg_u = ### 작성 필요 ###

        recalls.append(recall_u)
        precisions.append(precision_u)
        ndcgs.append(ndcg_u)

    recall = np.mean(recalls)
    precision = np.mean(precisions)
    ndcg = np.mean(ndcgs)
    return recall, precision, ndcg